# Tutorial

## Preparation

Prepare the folder structure:
- Create empty folders called `Data`, `Figure` and `Model` inside parent folder;
- Place the purchased orderbook data into `Data` folder. Purchase source: https://webshop.eex-group.com/epex-spot-public-market-data (Several data types are available. For example, the “Continuous Anonymous Orders History” for Germany costs 325 EUR/month.);
- Create your empty notebook ends with `.ipynb`;
- Simply run `pip install OrderFusion` in your notebook;


#### 📁 Project Structure

```text
├── Data/
│   └── Country (e.g. Germany)/
│       └── Intraday Continuous/
│           └── Orders/
│               └── Year (e.g. 2023)/
│                   ├── Month (e.g. 01)/
│                   ├── Month (e.g. 02)/
│                   ├── Month (e.g. 03)/
│                   └── ...
├── Figure/
├── Model/
├── Your_notebook.ipynb

## Setting

below define the forecasting target, interested period, hyperparameters, etc.

In [1]:
import OrderFusion
print(OrderFusion.__file__)

country = 'germany' # options: 'germany' or 'austria'
resolution = 'h' # options: 'h' for 60-min products, 'qh' for 15-min products
train_start, train_end = '2021-01-01', '2021-01-05'
val_start,   val_end   = '2021-01-06', '2021-01-07'
test_start,  test_end  = '2021-01-08', '2021-01-10'

#train_start, train_end = '2022-01-01', '2023-09-01'
#val_start,   val_end   = '2023-09-01', '2024-01-01'
#test_start,  test_end  = '2024-01-01', '2024-05-01'

#train_start, train_end = '2022-01-01', '2024-01-01'
#val_start,   val_end   = '2024-01-01', '2024-07-01'
#test_start,  test_end  = '2024-07-01', '2025-01-01'

indice = 'ID1' # options: 'ID1', 'ID2', 'ID3'

'''
only obtain most recent "num_trade" trades for data efficiency; 
could set to arbitrarily large value to use all trades,
but should larger than cutoff_len.
'''
num_trade = 128 

'''
assign the temporal weight of 1 to the most recent "cutoff_len" trades; 
distant trades are assigned weight of 0.
'''
cutoff_len = 16

'''
"pad_value" is used to fill the missing values when the number of trades is less than "num_trade". 
It should be outside the normal range of orderbook prices i.e. [-∞, -9999] or [9999, ∞].
'''
pad_value = 10000.0 

'''
target quantiles to forecast, could be many e.g. [0.05, 0.1, 0.5, 0.9, 0.95]
'''
quantiles = [0.1, 0.5, 0.9] 

hidden_dim, max_degree, num_heads, epoch, batch_size, seed = 16, 2, 1, 50, 256, 42
model_mode = 'OrderFusion'
show_progress_bar = True # options: True, False
save_path = OrderFusion.device_choice('local') # 'local' if run locally or 'cloud' run on Google Colab

# meta setting
setting = (country, resolution, indice, hidden_dim, max_degree, num_heads, epoch, batch_size, save_path, model_mode, [int(q * 100) for q in quantiles], cutoff_len, show_progress_bar)

/Users/work/PycharmProjects/OrderFusion/OrderFusion/__init__.py


# Step 1: extract input and output from orderbook

Separate 60-min and 15-min orderbook

In [2]:

import zipfile

file_path = "/Volumes/DATA/01_EPEX/Orders RAW/2021/01/Continuous_Orders-DE-20210101-20210721T083257000Z.csv.zip"
try:
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        print(f"ZIP file opened successfully. Contains: {file_list}")
except zipfile.BadZipFile:
    print(f"Error: '{file_path}' is not a valid zip file or is corrupted.")
except FileNotFoundError:
    print(f"Error: File '{file_path}' was not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")



ZIP file opened successfully. Contains: ['Continuous_Orders-DE-20210101-20210721T083257000Z.csv']


In [3]:
OrderFusion.filter_data(country, train_start, test_end) # --> CSV files of 60-min and 15-min products are separately saved in the 'data' folder

  processing germany 2021-01-01 → 2021-01-10:   0%|          | 0/10 [00:00<?, ?file/s]

Trying: /Volumes/DATA/01_EPEX/Orders RAW/2021/01/Continuous_Orders-DE-20210101-20210721T083257000Z.csv.zip
  Exists: True
  Size: 49218880 bytes


  processing germany 2021-01-01 → 2021-01-10:  10%|█         | 1/10 [00:09<01:29,  9.95s/file]

Trying: /Volumes/DATA/01_EPEX/Orders RAW/2021/01/Continuous_Orders-DE-20210102-20210721T084649000Z.csv.zip
  Exists: True
  Size: 49168681 bytes


  processing germany 2021-01-01 → 2021-01-10:  20%|██        | 2/10 [00:18<01:14,  9.37s/file]

...

  processing germany 2021-01-01 → 2021-01-10:  80%|████████  | 8/10 [01:15<00:17,  8.97s/file]

Trying: /Volumes/DATA/01_EPEX/Orders RAW/2021/01/Continuous_Orders-DE-20210109-20210721T092608000Z.csv.zip
  Exists: True
  Size: 49309956 bytes


  processing germany 2021-01-01 → 2021-01-10:  90%|█████████ | 9/10 [01:23<00:08,  8.71s/file]

Trying: /Volumes/DATA/01_EPEX/Orders RAW/2021/01/Continuous_Orders-DE-20210110-20210721T093048000Z.csv.zip
  Exists: True
  Size: 60792893 bytes


  processing germany 2021-01-01 → 2021-01-10: 100%|██████████| 10/10 [01:34<00:00,  9.45s/file]


Merge files per resolution and country

In [4]:
OrderFusion.merge_data(resolution, country)  # --> data are merged per resolution and country

Merging files:
  - raw_2021_01_h_germany.pkl


Extract input sequences per side

In [5]:
OrderFusion.get_input(resolution, country, indice)  # --> extracted sequences for buy side and sell side, respectively, are saved in the 'data' folder

  Extracting sequences: 100%|██████████| 240/240 [00:00<00:00, 359.55group/s, Processing date: 2021-01-10 22:00:00+00:00]


Extract output label (ID1, ID2, ID3)

In [6]:
OrderFusion.get_output(resolution, country, indice)  # --> extracted labels (ID1, ID2, ID3) are saved in the 'data' folder

  Extracting labels: 100%|██████████| 240/240 [00:00<00:00, 1292.11group/s]


Fit data scaler and save

In [7]:
OrderFusion.get_scaler(country, resolution, train_start, train_end)  # --> fitted scalers are saved in the 'data' folder

df_train shape: (548281, 5)
df_train head:
  Side             DeliveryStart                  TransactionTime  Price  \
0  BUY 2021-01-01 04:00:00+00:00 2020-12-31 15:14:25.113000+00:00   35.4   
1  BUY 2021-01-01 05:00:00+00:00 2020-12-31 15:14:25.427000+00:00   34.8   
2  BUY 2021-01-01 08:00:00+00:00 2020-12-31 15:14:25.740000+00:00   40.1   
3  BUY 2021-01-01 08:00:00+00:00 2020-12-31 15:14:52.916000+00:00   40.1   
4  BUY 2021-01-01 09:00:00+00:00 2020-12-31 15:14:26.228000+00:00   40.2   

   VolumeTraded  
0           1.9  
1           2.3  
2           2.7  
3           0.4  
4           3.0  
Price column null count: 0
Date range in data: 0 to 559307
Requested train range: 2021-01-01 to 2021-01-05
df before date filter shape: (973243, 5)
df DeliveryStart sample:
0   2021-01-01 04:00:00+00:00
1   2021-01-01 05:00:00+00:00
2   2021-01-01 08:00:00+00:00
3   2021-01-01 08:00:00+00:00
4   2021-01-01 09:00:00+00:00
5   2021-01-01 09:00:00+00:00
6   2021-01-01 10:00:00+00:00
7   2021-

# Step 2: read and preprocess input and output

In [8]:
save_path = OrderFusion.device_choice('local') 
orderbook_df = OrderFusion.read_data(save_path, country, resolution, indice)
orderbook_df.head()

,Sequence_Buy,SumVolume_x,NumTrades_x,Sequence_Sell,SumVolume_y,NumTrades_y,ID1,UTC
0,"[[50.1, 4.8, 5923.705], [50.1, 0.2, 5923.705],...",1941.1,818,"[[48.99, 1.5, 6143.81], [48.68, 1.0, 6137.543]...",1632.3,725,39.947916,2020-12-31 23:00:00+00:00
1,"[[43.99, 2.8, 6544.157], [44.02, 2.0, 6526.032...",2250.1,807,"[[44.5, 2.4000000000000004, 6777.644], [44.5, ...",2122.8,744,51.972644,2021-01-01 00:00:00+00:00
2,"[[47.8, 0.09999999999999987, 6108.985], [47.8,...",2256.2,868,"[[44.61, 5.0, 5763.528], [44.61, 5.0, 5763.528...",1881.5,785,88.791041,2021-01-01 01:00:00+00:00
3,"[[53.53, 0.20000000000000018, 5413.82], [54.51...",2227.6,975,"[[53.37, 1.3, 5202.482], [53.37, 1.5, 5202.482...",2024.6,957,62.851558,2021-01-01 02:00:00+00:00
4,"[[43.2, 0.1, 6342.986], [43.2, 0.1, 6267.905],...",2401.7,984,"[[41.22, 5.0, 5691.057], [41.2, 0.099999999999...",2329.0,1014,55.160259,2021-01-01 03:00:00+00:00


Split data intro training, validation, and testing

In [9]:
X_train, y_train, X_val, y_val, X_test, y_test = OrderFusion.split_data(orderbook_df, indice, train_start, train_end, val_start, val_end, test_start, test_end)

Scale data for each split

In [10]:
print(train_start, train_end)
print(test_end)
X_train, y_train, X_val, y_val, X_test, y_test = OrderFusion.scale_data(X_train, y_train, X_val, y_val, X_test, y_test, save_path, country, resolution)



2021-01-01 2021-01-05
2021-01-10
X_train_buy length: 96
X_train_sell length: 96
X_val_buy length: 24
X_val_sell length: 24
X_test_buy length: 48
X_test_sell length: 48


Pad data 

In [11]:
X_train, X_val, X_test = OrderFusion.pad_data(X_train, X_val, X_test, num_trade, pad_value)

# Step 3: train and optimize model

In [12]:
best_model, hist, paras_count = OrderFusion.optimize_model(X_train, y_train, X_val, y_val, setting)

paras: 1470

Epoch 1: val_loss improved from None to 1.51576, saving model to /Users/work/PycharmProjects/OrderFusion/Model/OrderFusion_germany_h_ID1.keras

Epoch 2: val_loss improved from 1.51576 to 1.50741, saving model to /Users/work/PycharmProjects/OrderFusion/Model/OrderFusion_germany_h_ID1.keras

Epoch 3: val_loss improved from 1.50741 to 1.49875, saving model to /Users/work/PycharmProjects/OrderFusion/Model/OrderFusion_germany_h_ID1.keras

Epoch 4: val_loss improved from 1.49875 to 1.48692, saving model to /Users/work/PycharmProjects/OrderFusion/Model/OrderFusion_germany_h_ID1.keras

Epoch 5: val_loss improved from 1.48692 to 1.47519, saving model to /Users/work/PycharmProjects/OrderFusion/Model/OrderFusion_germany_h_ID1.keras

Epoch 6: val_loss improved from 1.47519 to 1.46228, saving model to /Users/work/PycharmProjects/OrderFusion/Model/OrderFusion_germany_h_ID1.keras

Epoch 7: val_loss improved from 1.46228 to 1.44850, saving model to /Users/work/PycharmProjects/OrderFusion/

# Step 4: evaluate model's performance

In [13]:
OrderFusion.evaluate_model(best_model, X_test, y_test, quantiles, save_path, country, resolution)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step
AQL: 7.02, AQCR: 0.0, AIW: 119.01190185546875, AICR: 87.5, RMSE: 48.15, MAE: 15.15, R2: -13.16, Inference time: 0.3810739517211914s 



{'quantile_losses': [3.2080867626936573,
  7.576250018080103,
  10.288315275383681],
 'avg_quantile_loss': 7.02,
 'quantile_crossing_rate': 0.0,
 'median_quantile_rmse': 48.15,
 'median_quantile_mae': 15.15,
 'median_quantile_r2': -13.16,
 'inference_time': 0.3810739517211914,
 'y_test_original': array([46.81982437, 41.6995745 , 41.93731499, 39.92842848, 17.01060838,
        19.14068038, 66.13052192, 69.1242523 , 61.69383199, 64.57744659,
        64.38702305, 64.79794744, 64.11123526, 53.00391518, 74.10125039,
        64.61270461, 66.48437494, 66.87543065, 66.19955806, 61.69598727,
        53.50387731, 54.8706531 , 45.27217594, 45.6300375 , 52.35149051,
        53.05863623, 47.53154319, 53.40160965, 45.57160306, 45.29918238,
        50.99225755, 55.09664371, 54.34162383, 59.7800091 , 57.97034503,
        54.31380563, 54.17465596, 50.23198366, 52.70216478, 56.40022262,
        71.57579406, 58.05797341, 55.78644166, 35.0987905 , 35.59854917,
        27.50406589, 26.87209258, 46.82258667]

# Step 5: inference 

In [14]:
best_model = OrderFusion.load_best_model(quantiles, country, resolution, indice, save_path)
y_pred, y_test = OrderFusion.get_forecasts(best_model, save_path, X_test, y_test, quantiles, country, resolution)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step


# Optional: plotting forecasts versus true prices

In [15]:
OrderFusion.plot_forecasts(y_pred, y_test, 181, 400, indice)
OrderFusion.gif_conversion(indice)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/work/PycharmProjects/OrderFusion/Figure/ID1/ID1_0.png'